In [ ]:
import os
from google.colab import drive

os.system('pip install yt-dlp')
import yt_dlp


drive.mount('/content/drive', force_remount=True)

raw_audio_dir = "/content/drive/MyDrive/yourData"
os.makedirs(raw_audio_dir, exist_ok=True)

playlists = [
   #   "https://www.youtube.com/",
  #  "https://www.youtube.com/",                   CHOOSE YOUR CUSTOM AUDIO
   # "https://www.youtube.com/"
]




ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '192',
    }],

    'postprocessor_args': [
        '-ar', '16000',
        '-ac', '1'
    ],

    'outtmpl': f'{raw_audio_dir}/%(id)s.%(ext)s',
    'ignoreerrors': True,
    'quiet': False
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    for url in playlists:
        ydl.download([url])


In [ ]:
import os
os.system('pip install yt-dlp')
import yt_dlp

playlists = [
 #   "https://www.youtube.com/",
  #  "https://www.youtube.com/",
   # "https://www.youtube.com/"
]

ydl_opts = {'extract_flat': True, 'quiet': True}
total_duration = 0
video_count = 0



with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    for url in playlists:
        info = ydl.extract_info(url, download=False)
        if 'entries' in info:
            for entry in info['entries']:
                if entry and entry.get('duration'):
                    total_duration += entry['duration']
                    video_count += 1

hours = total_duration // 3600
minutes = (total_duration % 3600) // 60

print(f"\n إجمالي عدد الفيديوهات: {video_count} فيديو")
print(f" إجمالي الوقت التقريبي: {hours} ساعة و {minutes} دقيقة")

In [ ]:
!pip install faster-whisper pydub pandas tqdm
import os
import json
import pandas as pd
from faster_whisper import WhisperModel
from pydub import AudioSegment
from google.colab import drive
from tqdm import tqdm


drive.mount('/content/drive', force_remount=True)


raw_audio_dir = "/content/drive/MyDrive/allAudio"
dataset_dir = "/content/drive/MyDrive/AUDIO_dataset_After_TRANS"
wavs_dir = os.path.join(dataset_dir, "wavs")
os.makedirs(wavs_dir, exist_ok=True)
csv_path = os.path.join(dataset_dir, "metadata.csv")

resume_state_path = os.path.join(dataset_dir, "resume_state.json")


model = WhisperModel("large-v3", device="cpu", compute_type="int8")


existing_chunks = set()
last_text = ""
if not os.path.exists(csv_path):
    pd.DataFrame(columns=["audio_file", "text"]).to_csv(csv_path, index=False, sep="|", encoding='utf-8-sig')
else:
    try:
        df_existing = pd.read_csv(csv_path, sep="|")
        existing_chunks = set(df_existing["audio_file"].tolist())
        if not df_existing.empty:
            last_text = str(df_existing.iloc[-1]["text"])
    except pd.errors.EmptyDataError:
        pass


if os.path.exists(resume_state_path):
    with open(resume_state_path, "r", encoding="utf-8") as f:
        resume_state = json.load(f)
else:
    resume_state = {}

raw_files = [f for f in os.listdir(raw_audio_dir) if f.endswith(".wav")]

processed_log = os.path.join(dataset_dir, "processed_files.txt")
if os.path.exists(processed_log):
    with open(processed_log, "r", encoding="utf-8") as f:
        processed_files = f.read().splitlines()
else:
    processed_files = []

files_to_process = [f for f in raw_files if f not in processed_files]


for raw_file in tqdm(files_to_process, desc="Loading"):
    raw_path = os.path.join(raw_audio_dir, raw_file)


    file_state = resume_state.get(raw_file, {})
    current_offset_sec = file_state.get("last_end_time", 0.0)
    last_index = file_state.get("last_index", -1)

    try:
        audio = AudioSegment.from_wav(raw_path)


        prompt_to_use = last_text if (current_offset_sec > 0 and last_text) else None


        if current_offset_sec > 0:
            print(f"\n⏭ بنكمل فيديو {raw_file} من الدقيقة {current_offset_sec/60:.2f}...")

            audio_to_process = audio[int(current_offset_sec * 1000):]

            temp_path = os.path.join(dataset_dir, "temp_resume.wav")
            audio_to_process.export(temp_path, format="wav")
            target_file_for_whisper = temp_path
        else:
            audio_to_process = audio
            target_file_for_whisper = raw_path

        segments, info = model.transcribe(
            target_file_for_whisper,
            beam_size=5,
            language="ar",
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=500),
            initial_prompt=prompt_to_use
        )

        for i, segment in enumerate(segments):
            actual_index = last_index + 1 + i
            chunk_name = f"{raw_file.replace('.wav', '')}_{actual_index:04d}.wav"
            csv_audio_key = f"wavs/{chunk_name}"

            if csv_audio_key in existing_chunks:
                continue

            text = segment.text.strip()
            duration = segment.end - segment.start

            if len(text) > 0 and 1.5 <= duration <= 15.0:
                chunk_path = os.path.join(wavs_dir, chunk_name)

                start_ms = int(segment.start * 1000)
                end_ms = int(segment.end * 1000)
                chunk_audio = audio_to_process[start_ms:end_ms]

                chunk_audio.export(chunk_path, format="wav")

                new_row = pd.DataFrame([{"audio_file": csv_audio_key, "text": text}])
                new_row.to_csv(csv_path, mode='a', header=False, index=False, sep="|", encoding='utf-8-sig')

                existing_chunks.add(csv_audio_key)
                last_text = text
                resume_state[raw_file] = {
                    "last_end_time": current_offset_sec + segment.end,
                    "last_index": actual_index
                }
                with open(resume_state_path, "w", encoding="utf-8") as f:
                    json.dump(resume_state, f)

        with open(processed_log, "a", encoding="utf-8") as f:
            f.write(raw_file + "\n")

        if os.path.exists(os.path.join(dataset_dir, "temp_resume.wav")):
            os.remove(os.path.join(dataset_dir, "temp_resume.wav"))

    except Exception as e:
